# Testing Pandas for EBD Processing

Note: this data is pulling from an unzipped txt EBD file in the "ebd" folder not included in the repository

## Resources

- [Pandas vs R Cheatsheet](https://datascientyst.com/pandas-vs-r-cheat-sheet/)
- 

In [27]:
import pandas as pd
import json


# eBird Dataset file name
ebd_fn = 'ebd_US-NC_202101_202602_unv_smp_relMar-2026.txt'

## Load the EBD into a Pandas DF

In [28]:
ebd_raw = pd.read_csv(f'./ebd/{ebd_fn}', sep='\t')

C:\Users\skacl\AppData\Local\Temp\ipykernel_15620\4247824248.py:1: DtypeWarning: Columns (0: EXOTIC CODE, 1: AGE/SEX, 2: IBA CODE, 3: USFWS CODE, 4: OBSERVER ORCID ID, 5: PROJECT IDENTIFIERS) have mixed types. Specify dtype option on import or set low_memory=False.
  ebd_raw = pd.read_csv(f'./ebd/{ebd_fn}', sep='\t')


## Start Exploring

In [29]:
rows_columns = ebd_raw.shape
print(f'{rows_columns[0]:,d} rows, {rows_columns[1]} columns found')
# ebd['SAMPLING EVENT IDENTIFIER'].sort_values()

# get column names
columns =  ebd_raw.columns

# define observation fields
observation_fields = pd.Series([
  "GLOBAL UNIQUE IDENTIFIER",
  "TAXONOMIC ORDER",
  "CATEGORY",
  "TAXON CONCEPT ID",
  "COMMON NAME",
  "SCIENTIFIC NAME",
  "SUBSPECIES COMMON NAME",
  "SUBSPECIES SCIENTIFIC NAME",
  "EXOTIC CODE",
  "OBSERVATION COUNT",
  "BREEDING CODE",
  "BREEDING CATEGORY",
  "BEHAVIOR CODE",
  "AGE/SEX",
  "APPROVED",
  "REVIEWED",
  "REASON",
  "SPECIES COMMENTS"
])

# get checklist fields
checklist_fields = pd.Series(columns[~columns.isin(observation_fields)])

# add SAMPLING EVENT IDENTIFIER to observations, remove Unamed column from checklist_fields
observation_fields = pd.concat([observation_fields, pd.Series(['SAMPLING EVENT IDENTIFIER'])])
checklist_fields = checklist_fields[checklist_fields != 'Unnamed: 52']

print(f'checklist fields:\n{checklist_fields}')
print(f'observation fields:\n{observation_fields}')

20,052,478 rows, 53 columns found
checklist fields:
0              LAST EDITED DATE
1                       COUNTRY
2                  COUNTRY CODE
3                         STATE
4                    STATE CODE
5                        COUNTY
6                   COUNTY CODE
7                      IBA CODE
8                      BCR CODE
9                    USFWS CODE
10                  ATLAS BLOCK
11                     LOCALITY
12                  LOCALITY ID
13                LOCALITY TYPE
14                     LATITUDE
15                    LONGITUDE
16             OBSERVATION DATE
17    TIME OBSERVATIONS STARTED
18                  OBSERVER ID
19            OBSERVER ORCID ID
20    SAMPLING EVENT IDENTIFIER
21             OBSERVATION TYPE
22                PROTOCOL NAME
23                PROTOCOL CODE
24                PROJECT NAMES
25          PROJECT IDENTIFIERS
26             DURATION MINUTES
27           EFFORT DISTANCE KM
28               EFFORT AREA HA
29             NUMBE

## Nest Observations, create JSON

In [30]:
# create subset for testing
ebd = ebd_raw.head(1000)


In [35]:
# obs_only = ebd[observation_fields]
# check_only = ebd[checklist_fields]

# print(ebd.head(10))
# print(ebd.columns)
ebd_grouped = ebd.groupby('SAMPLING EVENT IDENTIFIER')
ebd_json = ebd_grouped.apply(
    lambda x: x.drop('SAMPLING EVENT IDENTIFIER',
    axis = 1
    ).to_dict(orient='records')
).to_json()
print(json.dumps(ebd_json, subset = 2))

KeyError: "['SAMPLING EVENT IDENTIFIER'] not found in axis"